In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np


NPZ_PATH = Path("/curie-home/zengjj/Renormalizer/multiset_ttn/P3HT:PCBM/multiset_treeX_bond_entropy/p3ht_ms_ttn_treeX_bond_entropy_32.npz")
TIME_FS = 200
OUT_DIR = Path("/curie-home/zengjj/Renormalizer/multiset_ttn/P3HT:PCBM/plot/multiset")


def short_node_label(raw):
    text = str(raw)
    if "Root" in text:
        return "Root"
    import re

    m = re.search(r"LE(\d+).*CS\1", text)
    if m:
        return f"LE/CS{m.group(1)}"
    m = re.search(r"OT(\d+)_m(\d+)", text)
    if m:
        return f"OT{m.group(1)}\nm{m.group(2)}"
    m = re.search(r"OT(\d+)-(\d+) and R", text)
    if m:
        return f"OT{m.group(1)}-{m.group(2)}\n+R"
    m = re.search(r"OT(\d+) and R", text)
    if m:
        return f"OT{m.group(1)}\n+R"
    m = re.search(r"OT(\d+)-(\d+) and F", text)
    if m:
        return f"OT{m.group(1)}-{m.group(2)}\n+F"
    m = re.search(r"OT(\d+)-(\d+)", text)
    if m:
        return f"OT{m.group(1)}-{m.group(2)}"
    if "F modes" in text:
        return "F\nmodes"
    m = re.search(r"OT(\d+).*modes", text)
    if m:
        return f"OT{m.group(1)}\nmodes"
    m = re.search(r"OT(\d+)", text)
    if m:
        return f"OT{m.group(1)}"
    if re.search(r"'R'", text):
        return "R"
    m = re.search(r"F(\d+)", text)
    if m:
        return f"F{m.group(1)}"
    return re.sub(r"[\[\]\(\)',]", "", text).replace("TreeX", "").strip() or text


def tree_layout(children, root=0):
    leaf_counts = {}

    def count_leaves(node):
        if node in leaf_counts:
            return leaf_counts[node]
        if len(children[node]) == 0:
            leaf_counts[node] = 1
        else:
            leaf_counts[node] = sum(count_leaves(child) for child in children[node])
        return leaf_counts[node]

    count_leaves(root)
    pos = {}
    max_depth = 0

    def assign(node, left, right, depth):
        nonlocal max_depth
        max_depth = max(max_depth, depth)
        pos[node] = ((left + right) / 2.0, -float(depth))
        cursor = left
        total = sum(leaf_counts[child] for child in children[node])
        for child in children[node]:
            width = (right - left) * leaf_counts[child] / total if total else 0.0
            assign(child, cursor, cursor + width, depth + 1)
            cursor += width

    assign(root, 0.0, float(leaf_counts[root]), 0)
    return pos, leaf_counts[root], max_depth


def format_entropy(value):
    if abs(value) < 1e-12:
        return "0"
    if abs(value) < 1e-3:
        return f"{value:.1e}"
    return f"{value:.3f}".rstrip("0").rstrip(".")


def format_bond_dim(value):
    try:
        return str(int(value))
    except (TypeError, ValueError):
        return str(value)


def format_edge_label(entropy, bond_dim):
    return f"{format_entropy(entropy)}\nχ={format_bond_dim(bond_dim)}"


def load_multiset_bond_entropy(npz_path, time_fs, entropy_key):
    npz_path = Path(npz_path).expanduser()
    npz = np.load(npz_path, allow_pickle=True)
    times = np.asarray(npz["time_fs"], dtype=float)
    if not (times.min() <= time_fs <= times.max()):
        raise ValueError(f"time_fs must be between {times.min()} and {times.max()}, got {time_fs}")

    requested_key = entropy_key
    if entropy_key not in npz.files and entropy_key == "S_cond" and "S_all" in npz.files:
        print(f"'S_cond' not found in {npz_path.name}; using equivalent 'S_all' instead")
        entropy_key = "S_all"
    if entropy_key not in npz.files:
        raise KeyError(f"{npz_path.name} does not contain {requested_key!r}")

    time_idx = int(np.argmin(np.abs(times - time_fs)))
    raw_labels = [str(x) for x in npz["dof_strs"]]
    adj = np.asarray(npz["adj_matrix"])
    state_labels = [str(x) for x in npz["state_labels"]] if "state_labels" in npz.files else []
    entropy_sets = np.asarray(npz[entropy_key], dtype=float)[time_idx]
    bond_dims = np.asarray(npz["bond_dims"], dtype=object)[time_idx]
    return {
        "npz_path": npz_path,
        "requested_key": requested_key,
        "actual_key": entropy_key,
        "time": float(times[time_idx]),
        "raw_labels": raw_labels,
        "adj": adj,
        "state_labels": state_labels,
        "entropy_sets": entropy_sets,
        "bond_dims": bond_dims,
    }


def draw_tree_bond_entropy_panel(ax, raw_labels, edges, pos, entropies, bond_dims, norm, cmap, title):
    edge_entropies = np.array([entropies[child] for _, child in edges], dtype=float)
    edge_bond_dims = [bond_dims[child] for _, child in edges]

    for (parent, child), entropy, bond_dim in zip(edges, edge_entropies, edge_bond_dims):
        x0, y0 = pos[parent]
        x1, y1 = pos[child]
        ax.plot(
            [x0, x1], [y0, y1],
            color=cmap(norm(entropy)),
            linewidth=0.52,
            solid_capstyle="round",
            zorder=1,
        )

        xm, ym = (x0 + x1) / 2.0, (y0 + y1) / 2.0
        dx, dy = x1 - x0, y1 - y0
        length = float(np.hypot(dx, dy))
        if length > 0:
            nx, ny = -dy / length, dx / length
            if ny < 0:
                nx, ny = -nx, -ny
        else:
            nx, ny = 0.0, 1.0
        ax.text(
            xm + 0.045 * nx,
            ym + 0.045 * ny,
            format_edge_label(float(entropy), bond_dim),
            ha="center",
            va="center",
            fontsize=1.55,
            color="#111111",
            bbox={"boxstyle": "round,pad=0.035", "facecolor": "white", "edgecolor": "none", "alpha": 0.68},
            zorder=2,
        )

    for idx, raw in enumerate(raw_labels):
        x, y = pos[idx]
        ax.text(
            x, y, short_node_label(raw),
            ha="center", va="center",
            fontsize=1.65,
            linespacing=0.82,
            bbox={"boxstyle": "round,pad=0.08", "facecolor": "white", "edgecolor": "#222222", "linewidth": 0.18},
            zorder=3,
        )

    ax.set_title(title, fontsize=7.0, pad=2.0)
    ax.set_axis_off()
    ax.margins(x=0.02, y=0.05)


def plot_multiset_treeX_bond_entropy(npz_path, time_fs, entropy_key, out_dir=OUT_DIR, show=True):
    data = load_multiset_bond_entropy(npz_path, time_fs, entropy_key)
    raw_labels = data["raw_labels"]
    adj = data["adj"]
    entropy_sets = data["entropy_sets"]
    bond_dims = data["bond_dims"]
    state_labels = data["state_labels"]

    children = {idx: list(np.flatnonzero(adj[idx])) for idx in range(len(raw_labels))}
    edges = [(parent, child) for parent, child_list in children.items() for child in child_list]
    pos, _, _ = tree_layout(children, root=0)
    all_edge_entropies = np.asarray([[entropies[child] for _, child in edges] for entropies in entropy_sets], dtype=float)
    global_vmax = max(float(np.nanmax(all_edge_entropies)), 1e-12)
    norm = mpl.colors.Normalize(vmin=0.0, vmax=global_vmax)
    cmap = plt.get_cmap("magma")

    nset = entropy_sets.shape[0]
    nrows, ncols = 13, 2
    fig, axes = plt.subplots(nrows, ncols, figsize=(22, 68), constrained_layout=False)
    axes = axes.ravel()

    for iset, ax in enumerate(axes):
        if iset >= nset:
            ax.set_axis_off()
            continue
        state_label = state_labels[iset] if iset < len(state_labels) else f"set{iset}"
        set_max = float(np.nanmax(all_edge_entropies[iset])) if all_edge_entropies.size else 0.0
        title = f"set {iset:02d}: {state_label}\nmax={set_max:.3g}"
        draw_tree_bond_entropy_panel(ax, raw_labels, edges, pos, entropy_sets[iset], bond_dims, norm, cmap, title)

    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes.tolist(), fraction=0.012, pad=0.006)
    cbar.set_label(data["requested_key"])

    fig.subplots_adjust(left=0.015, right=0.94, top=0.975, bottom=0.01, wspace=0.045, hspace=0.18)

    fig.suptitle(
        f"P3HT multiset TreeX {data['requested_key']}, all sets at time={data['time']:.3f} fs\n"
        "Panels are arranged as 2 columns x 13 rows and share the same color scale.",
        fontsize=15,
    )

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    max_bonddim = data["npz_path"].stem.rsplit("_", 1)[-1]
    out_pdf = out_dir / f"p3ht_multiset_treeX_{data['requested_key']}_all_sets_step{int(round(data['time']))}_M{max_bonddim}.pdf"
    fig.savefig(out_pdf)
    print(
        f"Saved {data['requested_key']} figure to {out_pdf}; "
        f"source key={data['actual_key']}, nset={nset}, shared_vmax={global_vmax:.6g}"
    )
    if show:
        plt.show()
    return fig, axes, out_pdf


fig, axes, out_pdf = plot_multiset_treeX_bond_entropy(NPZ_PATH, TIME_FS, "S_all_unnormed")


In [ ]:
fig, axes, out_pdf = plot_multiset_treeX_bond_entropy(NPZ_PATH, TIME_FS, "S_cond")


In [ ]:
# Plot Root and first-layer bond entropies for selected electronic sets.
# Run the first block above before this block so the helper functions are defined.

from matplotlib.patches import Rectangle

ROOT_LAYER_ENTROPY_KEY = "S_cond"
ROOT_LAYER_MAX_BONDDIM = 64
ROOT_LAYER_NPZ_PATH = NPZ_PATH.with_name(f"p3ht_ms_ttn_treeX_bond_entropy_{ROOT_LAYER_MAX_BONDDIM}.npz")
ROOT_LAYER_SETS = ["LE1", "LE13", "CS1", "CS13"]
ROOT_LAYER_OUT = OUT_DIR / f"p3ht_multiset_treeX_root_layer_{ROOT_LAYER_ENTROPY_KEY}_sets_step{TIME_FS}_M{ROOT_LAYER_MAX_BONDDIM}.pdf"

root_layer_data = load_multiset_bond_entropy(ROOT_LAYER_NPZ_PATH, TIME_FS, ROOT_LAYER_ENTROPY_KEY)
root_layer_labels = root_layer_data["raw_labels"]
root_layer_adj = root_layer_data["adj"]
root_layer_state_labels = root_layer_data["state_labels"]
root_layer_entropy_sets = root_layer_data["entropy_sets"]
root_layer_bond_dims = root_layer_data["bond_dims"]
root_child_indices = [int(i) for i in np.flatnonzero(root_layer_adj[0])]
root_child_names = ["OT1-2\n+ R", "OT3-6", "OT7-13\n+ F"]
left_second_indices = [int(i) for i in np.flatnonzero(root_layer_adj[root_child_indices[0]])]
right_second_indices = [int(i) for i in np.flatnonzero(root_layer_adj[root_child_indices[2]])]
left_third_indices = [int(i) for i in np.flatnonzero(root_layer_adj[left_second_indices[0]])]
left_second_names = ["OT1\n+ R", "OT2\nModes"]
left_third_names = ["OT1\nModes", "R"]
right_second_names = ["OT7-13", "F\nModes"]
selected_set_indices = [root_layer_state_labels.index(label) for label in ROOT_LAYER_SETS]


def draw_node_box(ax, center, size, text, facecolor, edgecolor="#062333", text_color="#111111", fontsize=12):
    width, height = size
    left = center[0] - width / 2
    bottom = center[1] - height / 2
    box = Rectangle(
        (left, bottom),
        width,
        height,
        facecolor=facecolor,
        edgecolor=edgecolor,
        linewidth=2.2,
        zorder=3,
    )
    ax.add_patch(box)
    ax.text(
        center[0],
        center[1],
        text,
        ha="center",
        va="center",
        color=text_color,
        fontsize=fontsize,
        zorder=4,
    )


def draw_entropy_label(ax, xy, xytext, entropy, bond_dim, rad=0.0, fontsize=10.5):
    ax.annotate(
        format_edge_label(float(entropy), bond_dim),
        xy=xy,
        xytext=xytext,
        ha="center",
        va="center",
        fontsize=fontsize,
        color="white",
        bbox={"boxstyle": "round,pad=0.24", "facecolor": "#9c9c9c", "edgecolor": "none", "alpha": 0.95},
        arrowprops={"arrowstyle": "->", "color": "#8f8f8f", "lw": 1.5, "connectionstyle": f"arc3,rad={rad}"},
        zorder=5,
    )


def draw_root_layer_panel(ax, set_idx, title):
    root_center = (0.5, 0.86)
    child_centers = [(0.20, 0.54), (0.50, 0.54), (0.80, 0.54)]
    left_second_centers = [(0.12, 0.30), (0.34, 0.30)]
    left_third_centers = [(0.04, 0.08), (0.20, 0.08)]
    right_second_centers = [(0.70, 0.24), (0.90, 0.24)]
    root_size = (0.24, 0.13)
    child_size = (0.24, 0.14)
    second_size = (0.18, 0.12)
    third_size = (0.13, 0.095)
    child_colors = ["#b9d8c8", "#ff7f2a", "#8e5aa0"]
    second_color = "#8e5aa0"

    root_bottom = (root_center[0], root_center[1] - root_size[1] / 2)
    for child_center in child_centers:
        child_top = (child_center[0], child_center[1] + child_size[1] / 2)
        ax.plot([root_bottom[0], child_top[0]], [root_bottom[1], child_top[1]], color="black", lw=2.3, zorder=1)

    draw_node_box(ax, root_center, root_size, "Root", "white", fontsize=13)
    for child_center, name, color in zip(child_centers, root_child_names, child_colors):
        draw_node_box(ax, child_center, child_size, name, color, text_color="white" if color != "#b9d8c8" else "#111111")

    entropies = root_layer_entropy_sets[set_idx]
    label_positions = [(0.18, 0.74), (0.50, 0.69), (0.82, 0.74)]
    arrow_targets = [(0.28, 0.62), (0.50, 0.62), (0.72, 0.62)]
    rads = [0.25, 0.0, -0.25]
    for child_idx, label_pos, arrow_target, rad in zip(root_child_indices, label_positions, arrow_targets, rads):
        draw_entropy_label(
            ax,
            arrow_target,
            label_pos,
            entropies[child_idx],
            root_layer_bond_dims[child_idx],
            rad=rad,
        )

    left_bottom_y = child_centers[0][1] - child_size[1] / 2
    right_bottom_y = child_centers[2][1] - child_size[1] / 2

    for second_center, name in zip(left_second_centers, left_second_names):
        second_top = (second_center[0], second_center[1] + second_size[1] / 2)
        ax.plot([child_centers[0][0], second_top[0]], [left_bottom_y, second_top[1]], color="black", lw=2.0, zorder=1)
        draw_node_box(ax, second_center, second_size, name, second_color, text_color="white", fontsize=10.5)

    for second_center, name in zip(right_second_centers, right_second_names):
        second_top = (second_center[0], second_center[1] + second_size[1] / 2)
        ax.plot([child_centers[2][0], second_top[0]], [right_bottom_y, second_top[1]], color="black", lw=2.0, zorder=1)
        draw_node_box(ax, second_center, second_size, name, second_color, text_color="white", fontsize=10.5)

    second_label_positions = [(0.06, 0.45), (0.36, 0.43), (0.66, 0.40), (0.94, 0.40)]
    second_arrow_targets = [(0.14, 0.37), (0.29, 0.38), (0.73, 0.33), (0.87, 0.33)]
    second_rads = [0.18, -0.18, 0.18, -0.18]
    for child_idx, label_pos, arrow_target, rad in zip(
        left_second_indices + right_second_indices,
        second_label_positions,
        second_arrow_targets,
        second_rads,
    ):
        draw_entropy_label(
            ax,
            arrow_target,
            label_pos,
            entropies[child_idx],
            root_layer_bond_dims[child_idx],
            rad=rad,
            fontsize=8.5,
        )

    ot1r_bottom_y = left_second_centers[0][1] - second_size[1] / 2
    for third_center, name in zip(left_third_centers, left_third_names):
        third_top = (third_center[0], third_center[1] + third_size[1] / 2)
        ax.plot([left_second_centers[0][0], third_top[0]], [ot1r_bottom_y, third_top[1]], color="black", lw=1.7, zorder=1)
        draw_node_box(ax, third_center, third_size, name, second_color, text_color="white", fontsize=8.3)

    third_label_positions = [(0.015, 0.205), (0.225, 0.205)]
    third_arrow_targets = [(0.065, 0.135), (0.175, 0.135)]
    third_rads = [0.18, -0.18]
    for child_idx, label_pos, arrow_target, rad in zip(
        left_third_indices,
        third_label_positions,
        third_arrow_targets,
        third_rads,
    ):
        draw_entropy_label(
            ax,
            arrow_target,
            label_pos,
            entropies[child_idx],
            root_layer_bond_dims[child_idx],
            rad=rad,
            fontsize=7.5,
        )

    ax.text(0.34, 0.08, "......", ha="center", va="center", fontsize=12)
    ax.text(0.80, 0.08, "......", ha="center", va="center", fontsize=12)
    ax.set_title(title, fontsize=21, fontweight="bold", pad=0)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect("equal")
    ax.axis("off")


fig, axes = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)
for ax, set_idx, set_label in zip(axes.ravel(), selected_set_indices, ROOT_LAYER_SETS):
    draw_root_layer_panel(ax, set_idx, set_label)

fig.suptitle(
    f"Root first-layer bond entropy at time={root_layer_data['time']:.3f} fs ({root_layer_data['requested_key']})",
    fontsize=16,
)
fig.savefig(ROOT_LAYER_OUT)
print(f"Saved figure to {ROOT_LAYER_OUT}")
plt.show()
